# TraceletCodeAgent on a single GAIA question

Cells 1-5 are identical to `traceletReAct.ipynb`, so the agent is configured exactly as the sweeps are.
Set `QUESTION_INDEX` below and run top to bottom. The trajectory streams live at `LogLevel.DEBUG`;
the last cell prints a compact per-step digest that flags repeated actions.


In [1]:
# Pick up edits to the editable-installed smolagents without a kernel restart.
%load_ext autoreload
%autoreload 2

# ===================== experiment configuration =====================
MODEL = "gpt-5.4-mini"                 # key of MODELS below
SKELETON_STRATEGY = "direct_prompt"     # "direct_prompt" | "post_process"
N_SAMPLES = 1                          # fill-in candidates per step; 1 = the ablation
USE_TRACELET_PROMPT = True             # tracelet_agent.yaml (template/fill-in protocol) vs default code_agent.yaml
QUESTION_INDEX = 112                 # single GAIA validation index to run
N_QUESTIONS = None                   # unused here; kept so the config block matches the sweep notebook
MAX_STEPS = 50
RUN_TAG = "_v5"                   # e.g. "_rerun" for fresh output paths
# ====================================================================

MODELS = {
    "gpt-4o": "openai",
    "gpt-5.4-mini": "openai",
    "Qwen/Qwen3.5-9B": "together",
    "Qwen/Qwen3.7-Plus": "together",
}
ENABLE_THINKING = False  # Together-served open-weight models only

assert MODEL in MODELS, f"unknown MODEL {MODEL!r}; pick one of {list(MODELS)}"
assert SKELETON_STRATEGY in ("direct_prompt", "post_process")

In [ ]:
import os
import sys
sys.path.insert(0, "../examples/open_deep_research")

from dotenv import load_dotenv
load_dotenv()

from smolagents import OpenAIModel

model_name = MODEL
provider = MODELS[MODEL]

# Only Qwen3.7-Plus rejects non-streaming, and streaming drops fill-in logprobs, so stream only when forced.
STREAM_ONLY = {"Qwen/Qwen3.7-Plus"}

if provider == "openai":
    model = OpenAIModel(model_id=model_name, api_key=os.environ["OPENAI_API_KEY"])
else:
    # Together needs enable_thinking nested in chat_template_kwargs for vLLM-served models (see naiveReAct).
    model = OpenAIModel(
        model_id=model_name,
        api_base="https://api.together.ai/v1/",
        api_key=os.environ["TOGETHER_API_KEY"],
        extra_body={"chat_template_kwargs": {"enable_thinking": ENABLE_THINKING}},
        client_kwargs={"timeout": 300.0},  # bound a stalled stream instead of hanging forever
    )
stream_outputs = model_name in STREAM_ONLY

print(f"{model_name} via {provider} | stream_outputs={stream_outputs}")

In [3]:
# Standard GAIA tool stack, same as naiveReAct.ipynb.
from common_setup import build_tools

tools, ti_tool, visualizer = build_tools(model)

/Users/poorvag/Work/smolagents/.venv/lib/python3.14/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [4]:
import importlib.resources

import yaml

from smolagents.monitoring import LogLevel
from smolagents.tracelet_agent import TraceletCodeAgent

# None lets CodeAgent fall back to its default code_agent.yaml.
prompt_templates = None
if USE_TRACELET_PROMPT:
    prompt_templates = yaml.safe_load(
        importlib.resources.files("smolagents.prompts").joinpath("tracelet_agent.yaml").read_text()
    )

agent = TraceletCodeAgent(
    tools=tools,
    model=model,
    max_steps=MAX_STEPS,
    verbosity_level=LogLevel.DEBUG,
    additional_authorized_imports=["pandas", "numpy", "PIL", "json", "io", "zipfile", "csv", "openpyxl"],
    n_samples=N_SAMPLES,
    skeleton_strategy=SKELETON_STRATEGY,
    stream_outputs=stream_outputs,
    prompt_templates=prompt_templates,
)

In [5]:
# Load GAIA validation set from HuggingFace
import pandas as pd
from common_setup import load_gaia_dataset

SET_TO_RUN = "validation"
eval_ds = load_gaia_dataset(set_to_run=SET_TO_RUN)

print(f"Loaded {len(eval_ds)} examples")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loaded 165 examples


In [6]:
# Run one GAIA question. Full model output for every step prints live at LogLevel.DEBUG.
from common_setup import question_scorer

example = eval_ds.to_list()[QUESTION_INDEX]
assert not example["file_name"], f"q{QUESTION_INDEX} has an attachment; attachment preprocessing is skipped here"

print(f"Q{QUESTION_INDEX}  task_id={example.get('task_id')}")
print(f"question : {example['question']}")
print(f"true     : {example['true_answer']}")
print("=" * 100)

prediction = agent.run(example["question"])

print("=" * 100)
print(f"prediction : {prediction!r}")
print(f"true       : {example['true_answer']!r}")
print(f"correct    : {question_scorer(str(prediction), example['true_answer'])}")


Q112  task_id=305ac316-eef6-4446-960a-92d80d542f82
question : Who did the actor who played Ray in the Polish-language version of Everybody Loves Raymond play in Magda M.? Give only the first name.
true     : Wojciech


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Who did the actor who played Ray in the Polish-language version of Everybody Loves Raymond play in Magda M.?    │
│ Give only the first name.                                                                                       │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2 memory messages


Thought: ──────────────────────────────────────────────────────────────────────────────────────────────────────────
Thought: I’ll identify the Polish-language actor who voiced/played Ray in the Polish version of Everybody Loves    
Raymond, then find his role in Magda M., and return only the first name. I’ll use web search for the cast          
information and then verify the Magda M. role.                                                                     

...

[Tracelet] no tool-call arguments to sample this step -- direct execution (no fill-in sampling, no judge).

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Petey")                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Petey

[Step 4: Duration 1.03 seconds| Input tokens: 45,943 | Output tokens: 476]

prediction : 'Petey'
true       : 'Wojciech'
correct    : False


In [ ]:
# Per-step digest of the run above; repeated actions are what to look for.
import re

from smolagents.memory import ActionStep

DIGEST_PATH = f"q{QUESTION_INDEX}_trajectory.txt"

steps = [s for s in agent.memory.steps if isinstance(s, ActionStep)]
first_seen, dups, out = {}, 0, []

for k, s in enumerate(steps):
    code = (s.code_action or "").strip()
    sig = re.sub(r"\s+", " ", code)
    tag = ""
    if code:
        if sig in first_seen:
            tag = f"   <-- IDENTICAL CODE TO STEP {first_seen[sig]}"
            dups += 1
        else:
            first_seen[sig] = k
    m = re.search(r"Thought:(.*?)(?:<code>|$)", s.model_output or "", re.S)
    thought = m.group(1).strip() if m else ""
    out.append("-" * 100)
    out.append(f"STEP {k}   input_tokens={s.token_usage.input_tokens:,}{tag}" if s.token_usage else f"STEP {k}{tag}")
    if s.error:
        out.append(f"  ERROR {type(s.error).__name__}: {str(s.error)[:300]}")
    out.append(f"  THOUGHT: {thought[:300] or '(none)'}")
    out.append(f"  CODE   : {code[:400] or '(none)'}")
    out.append(f"  OBS    : {str(s.observations or '')[:400].strip() or '(empty)'}")

text = "\n".join(out)
print(text)
print("=" * 100)
print(f"{len(steps)} action steps | {dups} steps repeated an earlier step's code verbatim")
with open(DIGEST_PATH, "w") as f:
    f.write(text)
print(f"written to baseline/{DIGEST_PATH}")
